In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("DADS MP2 Dataset.csv")

In [13]:
print("Original transactions:", len(df))

df = df.drop_duplicates()

df["Date"] = pd.to_datetime(df["Date"],errors="coerce",dayfirst=True)

df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

df["Type"] = df["Type"].astype(str).str.lower().str.strip()

df["Type"] = df["Type"].replace({
    "dr": "debit",
    "debit": "debit",
    "cr": "credit",
    "credit": "credit"
})

df = df.dropna(subset=["Date", "Amount"])

df["hour"] = pd.to_numeric(
    df["Time"].astype(str).str[:2],
    errors="coerce"
)

df["month"] = df["Date"].dt.month_name().str[:3]

print("Clean transactions:", len(df))

Original transactions: 1328
Clean transactions: 143


In [14]:
# 2. VENDOR EXTRACTION

vendor_keywords = {

    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT"],
    "Amazon": ["AMAZON", "AMZN"],
    "Myntra": ["MYNTRA"],
    "Flipkart": ["FLIPKART"],
    "Ajio": ["AJIO"],
    "Meesho": ["MEESHO"],

    "Zerodha": ["ZERODHA", "COIN"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "BookMyShow": ["BOOKMYSHOW"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "YouTube": ["YOUTUBE"],
    "Google": ["GOOGLE"],

    "Cafe Coffee Day": ["CCD"],
    "Starbucks": ["STARBUCKS"],
    "McDonalds": ["MCDONALD"],
    "KFC": ["KFC"],
    "Dominos": ["DOMINO"],

    "HP Petrol": ["HPCL", "HP PETROL"],
    "Indian Oil": ["INDIAN OIL", "IOCL"],

    "Electricity": ["ELECTRICITY", "BESCOM"],
    "Airtel": ["AIRTEL"],
    "Jio": ["JIO"],

    "Personal Transfer": ["UPI-PRIYA", "UPI-ANKIT"],
    "Cash Withdrawal": ["ATM-WDL", "ATM WITHDRAWAL"]
}

In [15]:
def extract_vendor(description):
    description = str(description).upper()

    if "ATM-WDL" in description:
        return "Cash Withdrawal"

    if "UPI-PRIYA" in description or "UPI-ANKIT" in description:
        return "Personal Transfer"

    for vendor, keywords in vendor_keywords.items():
        for word in keywords:
            if word in description:
                return vendor
    return "Uncategorised"
df["vendor_clean"] = df["Description"].apply(extract_vendor)
print("\nTop Vendors:")
print(df["vendor_clean"].value_counts().head(10))


Top Vendors:
vendor_clean
Uncategorised    46
Swiggy           28
Zomato           16
Ola               8
Uber              7
Zepto             5
Flipkart          4
Starbucks         4
Zerodha           4
Amazon            3
Name: count, dtype: int64


In [16]:
# 3. CATEGORY TAGGING

category_map = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",

    "Amazon": "E-commerce",
    "Myntra": "E-commerce",
    "Flipkart": "E-commerce",
    "Ajio": "E-commerce",
    "Meesho": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",

    "Cafe Coffee Day": "Cafe",
    "Starbucks": "Cafe",

    "McDonalds": "Restaurants",
    "KFC": "Restaurants",
    "Dominos": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "YouTube": "Subscriptions",

    "Electricity": "Utilities",
    "Airtel": "Utilities",
    "Jio": "Utilities",

    "Groceries": "Groceries",

    "Zerodha": "Investments",

    "HP Petrol": "Fuel",
    "Indian Oil": "Fuel",

    "BookMyShow": "Entertainment",

    "Personal Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}

In [17]:
df["category"] = df["vendor_clean"].map(category_map)

df["category"] = df["category"].fillna("Uncategorised")

print("\nCategories:")
print(df["category"].value_counts())


Categories:
category
Uncategorised        46
Food Delivery        44
Transport            17
E-commerce            9
Quick Commerce        7
Cafe                  6
Investments           4
Entertainment         3
Cash Withdrawal       3
Personal Transfer     2
Subscriptions         1
Fuel                  1
Name: count, dtype: int64


In [18]:
# 4. SPENDING OVERVIEW
credits = df.loc[df["Type"] == "credit", "Amount"].sum()  # AI assisted code
debits = df.loc[df["Type"] == "debit", "Amount"].sum()
net_change = credits - debits
savings_rate = (
    net_change / credits * 100
    if credits != 0 else 0
)

print("\n" + "=" * 60)
print("              SpendDNA REPORT")
print("=" * 60)

print("\nEXECUTIVE SUMMARY")

print(f"Total Credits : ₹{credits:,.2f}")
print(f"Total Debits  : ₹{debits:,.2f}")
print(f"Net Change    : ₹{net_change:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")
print(f"Transactions  : {len(df)}")
print(f"Vendors       : {df['vendor_clean'].nunique()}")


              SpendDNA REPORT

EXECUTIVE SUMMARY
Total Credits : ₹170,094.00
Total Debits  : ₹205,310.00
Net Change    : ₹-35,216.00
Savings Rate  : -20.70%
Transactions  : 143
Vendors       : 19


In [19]:
# 5. TOP CATEGORIES

debit_df = df[df["Type"] == "debit"]
category_spend = (debit_df.groupby("category")["Amount"].sum().sort_values(ascending=False))
print("\nTOP CATEGORIES")
for category, amount in category_spend.head(10).items():
    percentage = amount / debits * 100
    bars = "#" * int(percentage / 2)
    print(f"{category:<20} "f"{bars:<25} "f"{percentage:5.1f}%  ₹{amount:,.0f}")


TOP CATEGORIES
Uncategorised        ##################         37.3%  ₹76,627
Investments          ##############             29.2%  ₹60,000
E-commerce           #######                    15.2%  ₹31,220
Food Delivery        ####                        9.6%  ₹19,617
Transport            #                           2.5%  ₹5,145
Quick Commerce                                   1.5%  ₹3,094
Cash Withdrawal                                  1.5%  ₹3,000
Cafe                                             1.0%  ₹2,058
Entertainment                                    0.8%  ₹1,740
Personal Transfer                                0.7%  ₹1,512


In [20]:
# 6. TOP VENDORS

vendor_spend = (debit_df.groupby("vendor_clean")["Amount"].sum().sort_values(ascending=False))
print("\nTOP VENDORS")
for vendor, amount in vendor_spend.head(10).items():
    count = len(debit_df[debit_df["vendor_clean"] == vendor])

    print(f"{vendor:<20} "f"₹{amount:,.0f} "f"({count} transactions)")


TOP VENDORS
Uncategorised        ₹76,627 (44 transactions)
Zerodha              ₹60,000 (4 transactions)
Myntra               ₹13,629 (2 transactions)
Swiggy               ₹12,638 (28 transactions)
Flipkart             ₹10,041 (4 transactions)
Amazon               ₹7,550 (3 transactions)
Zomato               ₹6,979 (16 transactions)
Cash Withdrawal      ₹3,000 (3 transactions)
Uber                 ₹2,650 (7 transactions)
Zepto                ₹2,392 (5 transactions)


In [21]:
# 7. MONTHLY TREND

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
monthly = (debit_df.groupby(["category", "month"])["Amount"].sum().unstack(fill_value=0))
monthly = monthly.reindex(columns=month_order, fill_value=0)
print("\nMONTHLY SPENDING TREND")
print(monthly.round(0))


MONTHLY SPENDING TREND
month                 Jan     Feb      Mar     Apr      May     Jun
category                                                           
Cafe                295.0   355.0      0.0   447.0    166.0     0.0
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0
E-commerce         5346.0   996.0   2147.0  3311.0      0.0  3000.0
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0
Fuel                  0.0     0.0      0.0     0.0      0.0     0.0
Investments           0.0     0.0      0.0     0.0      0.0     0.0
Personal Transfer     0.0     0.0      0.0     0.0      0.0     0.0
Quick Commerce      825.0   697.0      0.0     0.0    502.0     0.0
Subscriptions         0.0     0.0      0.0     0.0      0.0     0.0
Transport           826.0   621.0    439.0     0.0    431.0   134.0
Uncategorised      6500.0   267.0  21130.0   513.0  18753.0  4522.0


In [22]:
# 8. TIME-OF-DAY ANALYSIS

print("\nTIME-OF-DAY PATTERNS")
food = debit_df[debit_df["category"] == "Food Delivery"]
late_night = food[(food["hour"] >= 21) | (food["hour"] <= 2)]

if len(food) > 0:
    late_percentage = len(late_night) / len(food) * 100
    print(f"Food Delivery orders from 9 PM - 2 AM: "f"{late_percentage:.1f}%")

time_matrix = pd.pivot_table(debit_df,values="Amount",index="category",columns="hour",aggfunc="sum",fill_value=0)
print("\nCategory x Hour Spending:")
print(time_matrix.round(0))


TIME-OF-DAY PATTERNS
Food Delivery orders from 9 PM - 2 AM: 25.0%

Category x Hour Spending:
hour                  0       1      2       3        4       5       6   \
category                                                                   
Cafe                 0.0     0.0    0.0     0.0      0.0     0.0     0.0   
Cash Withdrawal      0.0     0.0    0.0     0.0      0.0     0.0     0.0   
E-commerce           0.0     0.0    0.0  2462.0      0.0     0.0     0.0   
Entertainment      562.0     0.0    0.0     0.0      0.0     0.0     0.0   
Food Delivery      394.0     0.0  318.0     0.0    413.0   537.0     0.0   
Fuel                 0.0     0.0    0.0     0.0      0.0     0.0     0.0   
Investments          0.0     0.0    0.0     0.0  15000.0     0.0     0.0   
Personal Transfer    0.0     0.0    0.0     0.0      0.0     0.0     0.0   
Quick Commerce       0.0     0.0  697.0     0.0      0.0     0.0     0.0   
Subscriptions        0.0     0.0    0.0     0.0      0.0     0.0     0

In [23]:
# 9. ANOMALY DETECTION

category_mean = (debit_df.groupby("category")["Amount"].transform("mean"))
category_std = (debit_df.groupby("category")["Amount"].transform("std"))
debit_df = debit_df.copy()
debit_df["z_score"] = ((debit_df["Amount"] - category_mean)/ category_std)
anomalies = debit_df[debit_df["z_score"] > 2].sort_values("z_score",ascending=False)
print("\nTOP ANOMALIES")

for _, row in anomalies.head(10).iterrows():

    print(
        f"{row['Date'].strftime('%d-%b')} | "
        f"{row['vendor_clean']:<18} | "
        f"{row['category']:<18} | "
        f"₹{row['Amount']:,.0f} | "
        f"z = {row['z_score']:.2f}"
    )


TOP ANOMALIES
05-May | Uncategorised      | Uncategorised      | ₹18,000 | z = 4.41
06-Mar | Uncategorised      | Uncategorised      | ₹18,000 | z = 4.41
01-Sep | Myntra             | E-commerce         | ₹10,745 | z = 2.53


In [27]:
# 10. SPENDING ARCHETYPES       #AI Assisted

print("\n" + "=" * 60)
print("          RAHUL'S SPENDING ARCHETYPES")
print("=" * 60)
archetypes = []
category_percentage = (category_spend / debits * 100)

food_categories = ["Food Delivery","Restaurants","Cafe"]
food_spend = category_spend[category_spend.index.isin(food_categories)].sum()
food_percentage = food_spend / debits * 100
if food_percentage > 25:
    archetypes.append("THE FOODIE")
    print(f"-> THE FOODIE "f"({food_percentage:.1f}% spent on food)")
quick_percentage = (category_spend.get("Quick Commerce", 0)/ debits * 100)
if quick_percentage > 15:
    archetypes.append("THE QUICK COMMERCE JUNKIE")
    print(f"-> THE QUICK COMMERCE JUNKIE "f"({quick_percentage:.1f}%)")

shopping_percentage = (category_spend.get("E-commerce", 0)/ debits * 100)
if shopping_percentage > 15:
    archetypes.append("THE SHOPAHOLIC")
    print(f"-> THE SHOPAHOLIC "f"({shopping_percentage:.1f}%)")

investment_percentage = (category_spend.get("Investments", 0)/ debits * 100)

if investment_percentage > 15:
    archetypes.append("THE INVESTOR")
    print(f"-> THE INVESTOR "f"({investment_percentage:.1f}%)")

if len(food) > 0:
    late_food_percentage = (len(late_night) / len(food) * 100)
    if late_food_percentage > 50:
        archetypes.append("THE LATE-NIGHT SNACKER")
        print(f"-> THE LATE-NIGHT SNACKER "f"({late_food_percentage:.1f}% late-night food)")

transport_percentage = (category_spend.get("Transport", 0)/ debits * 100)
if transport_percentage > 10:
    archetypes.append("THE CAB COMMUTER")
    print(f"-> THE CAB COMMUTER "f"({transport_percentage:.1f}%)")

if savings_rate < 10:
    archetypes.append("THE YOLO SPENDER")
    print(f"-> THE YOLO SPENDER "f"(Savings rate: {savings_rate:.1f}%)")


          RAHUL'S SPENDING ARCHETYPES
-> THE SHOPAHOLIC (15.2%)
-> THE INVESTOR (29.2%)
-> THE YOLO SPENDER (Savings rate: -20.7%)


In [30]:
# DISCIPLINED SAVER
if savings_rate > 40:
    archetypes.append("THE DISCIPLINED SAVER")
    print(f"-> THE DISCIPLINED SAVER "f"(Savings rate: {savings_rate:.1f}%)")

In [29]:
# 11. KEY INSIGHTS      # AI Assisted
print("\n" + "=" * 60)
print("                    KEY INSIGHTS")
print("=" * 60)
top_category = category_spend.index[0]
top_category_amount = category_spend.iloc[0]
print(f"1. Highest spending category: {top_category} "f"with ₹{top_category_amount:,.0f}.")
print(f"2. Savings rate is {savings_rate:.1f}%, "f"showing the difference between credits and debits.")
print(f"3. {len(anomalies)} transactions were detected "f"as category-wise anomalies.")
print("\nArchetypes detected:")
for archetype in archetypes:
    print(" -", archetype)

print("\n" + "=" * 60)
print("                 END OF REPORT")
print("=" * 60)


                    KEY INSIGHTS
1. Highest spending category: Uncategorised with ₹76,627.
2. Savings rate is -20.7%, showing the difference between credits and debits.
3. 3 transactions were detected as category-wise anomalies.

Archetypes detected:
 - THE SHOPAHOLIC
 - THE INVESTOR
 - THE YOLO SPENDER

                 END OF REPORT
